In [5]:
# setting the environment variables, the keys
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

from config import set_environment

# for the keys - as explained early in chapter 2
set_environment()

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=1.0)

# Streaming

Firts, let's download a public MMLU [dataset](https://huggingface.co/datasets/cais/mmlu) from HuggingFace:

In [7]:
from datasets import load_dataset
ds = load_dataset("cais/mmlu", "high_school_geography")

Now, let's create a simple research agent:

In [8]:
from langchain_community.agent_toolkits.load_tools import load_tools
from langgraph.prebuilt import create_react_agent

research_tools = load_tools(
    tool_names=["ddg-search", "arxiv", "wikipedia"],
    llm=llm,
)

system_prompt = (
    "You're a hard-working, curious and creative student. "
    "You're working on an exam question. Think step by step."
    "Always provide an argumentation for your answer. "
    "Do not assume anything, use available tools to search "
    "for evidence and supporting statements."
)

In [10]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langgraph.graph import MessagesState
from langgraph.prebuilt.chat_agent_executor import AgentState

raw_prompt_template = (
    "Answer the following multiple-choice question. "
    "\nQUESTION:\n{question}\n\nANSWER OPTIONS:\n{options}\n"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("user", raw_prompt_template),
        ("placeholder", "{messages}")
    ]
)

class ResearchState(AgentState):
    question: str
    options: str

research_agent = create_react_agent(model=llm, tools=research_tools, state_schema=ResearchState, prompt=prompt)

Now we need to prepare a _question` and answer _options_:

In [11]:
i = 6
ds_dict = ds["test"].take(100).to_dict()
question = ds_dict["question"][i]
options = "\n".join([f"{i}. {a}" for i, a in enumerate(ds_dict["choices"][i])])

In [13]:
print(question)
print(options)

Which of the following countries does NOT have a well-known example of a relict boundary?
0. Vietnam
1. United Kingdom
2. Germany
3. Bolivia


Compare outputs (and available amount of messages in the `event` instance) when using `stream_mode=values` vs `updates`:

In [12]:
async for _, event in research_agent.astream({"question": question, "options": options}, stream_mode=["values"]):
    print(len(event["messages"]))

0
1
2
3
4
5
6
7
8
9
10
11
12
13


In [14]:
async for _, event in research_agent.astream({"question": question, "options": options}, stream_mode=["updates"]):
  node = list(event.keys())[0]
  print(node, len(event[node].get("messages", [])))

agent 1
tools 1
agent 1
tools 1
agent 1
tools 1
agent 1
tools 1
agent 1
tools 1
agent 1


In [15]:
async for _, event in research_agent.astream({"question": question, "options": options}, stream_mode=["updates"]):
  print(event)

{'agent': {'messages': [AIMessage(content='', additional_kwargs={'function_call': {'name': 'duckduckgo_search', 'arguments': '{"query": "relict boundary definition"}'}, '__gemini_function_call_thought_signatures__': {'fe6c44e9-85b7-45e7-ac4f-419190142b67': 'CqcGAXLI2nxfOxLQThW5ChcenpI+KdajL79UR2Ds5ZqsRzQ06VNhtaQNcqEUrJALeb5KD2WvwPvtRkTfOE2uUHMKiNiZAo+ztz92Zg+QC7GCGOj50eezKk2HRSfDAPMgd9qJZGQo4rRFFbQ3nnVItuK2iJIXniSyJi13qjvkWNRjTUY5bHiq3WbOgvrqZDL9b34vgN2bFxkgG+L3ECC1dKFYTlGqEVYotst1s1qQSMR1hmPlOJMkKaUPqesGPJKV6/iOpYbGMmoHgd9VeuY1grzazKa+8kCtyPyESelkldriC5oIiYgQauhPpFiDyTpfGd6BW/m7nNjeZ501450vH16P07BJwTy6oA+oGOJ2lw+LxzrsLus2lqHBWEN680Fb52zvDo2Bgfhs/r9cP4MbM4mld/hYe1zp8sUXY1TNuWwjHAIjAhPpaoWMk1QSXUvGv7Q4qiOClpuNQceKsxvXpLEmOELFG4ZNsiNbCLjOoWE8xwJxiFIukCBiR+U1X/iedN5Np7I4M9lqkOWIrDIjiC5WKnpo04ngws/RdNWPUuRkhONqhn4UZp/uoUkocgH8K0Mme261FHpHZ/ICmslCXhYFYJuDA9KDWEuxyH+AUJ0vxddZBsJh8/vyLB4alyiHUxofJk9aninpx9dNDvhyyKsF5bqTnf567MaSBnVKA0BnHss+i1wBuNnDHkvZWUz0RgRrMWIqF1aOg5Kbo2ZWhpap6zbPNfZuOpAU18

We can also explore what types of events we have seen:

In [16]:
seen_events = set([])
async for event in research_agent.astream_events({"question": question, "options": options}, version="v1"):
  if event["event"] not in seen_events:
    seen_events.add(event["event"])

print(seen_events)

{'on_tool_end', 'on_prompt_end', 'on_tool_start', 'on_chat_model_end', 'on_chain_start', 'on_chain_stream', 'on_chain_end', 'on_chat_model_stream', 'on_chat_model_start', 'on_prompt_start'}
